# 오토인코더 실습

**Autoencoder · AE**

입력을 저차원으로 압축한 뒤 다시 복원하도록 학습해 표현을 얻는 신경망.

소재 분야에서 이해하기: 스펙트럼을 압축해 잠재 표현으로 이상 시료를 찾는다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [변분 오토인코더 원논문](https://arxiv.org/abs/1312.6114)

## 1. 압축과 복원

스펙트럼을 흉내낸 데이터를 2차원 병목으로 압축한 뒤 복원합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

grid = np.linspace(0, 1, 40)

def spectra(n, seed=0):
    local = np.random.default_rng(seed)
    centre = local.uniform(0.25, 0.75, n)
    width = local.uniform(0.04, 0.12, n)
    data = np.exp(-((grid[None, :] - centre[:, None]) ** 2) / (2 * width[:, None] ** 2))
    return data + local.normal(0, 0.02, data.shape), centre, width

data, centre, width = spectra(600, 0)
for row in data[:4]:
    plt.plot(grid, row)
plt.xlabel('channel'); plt.ylabel('intensity'); plt.title('synthetic spectra'); plt.show()

In [ ]:
from sklearn.neural_network import MLPRegressor

autoencoder = MLPRegressor(hidden_layer_sizes=(16, 2, 16), max_iter=4000,
                           random_state=0, tol=1e-7).fit(data, data)
reconstructed = autoencoder.predict(data)
print('복원 평균 절대 오차 %.4f (신호 최대값 1.0 기준)' % np.mean(np.abs(reconstructed - data)))

plt.plot(grid, data[0], label='input')
plt.plot(grid, reconstructed[0], '--', label='reconstruction')
plt.legend(); plt.xlabel('channel'); plt.show()

## 2. 병목이 무엇을 배웠는지 확인

병목 2차원 값이 실제 물리 인자(피크 위치·폭)와 대응되는지 봅니다.

In [ ]:
def bottleneck(model, X, layer=2):
    activation = X
    for index, (weight, bias) in enumerate(zip(model.coefs_, model.intercepts_), 1):
        activation = activation @ weight + bias
        if index == layer:
            return activation
        activation = np.maximum(activation, 0)
    return activation

latent = bottleneck(autoencoder, data)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for axis, (values, name) in zip(axes, [(centre, 'peak position'), (width, 'peak width')]):
    scatter = axis.scatter(latent[:, 0], latent[:, 1], c=values, cmap='viridis', s=8)
    axis.set_title('latent space coloured by ' + name)
    axis.set_xlabel('latent 1'); axis.set_ylabel('latent 2')
    plt.colorbar(scatter, ax=axis)
plt.tight_layout(); plt.show()
print('색이 방향에 따라 정렬되어 있으면 병목이 그 인자를 담고 있다는 뜻입니다.')

## 3. 해석

40차원 스펙트럼을 2차원으로 줄여도 복원이 되는 것은 데이터가 사실상 2개의 인자로 만들어졌기
때문입니다. 이 잠재 표현은 이상 시료 탐지나 시각화에 쓸 수 있습니다.

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#autoencoder)을 여세요.